# 투구 제구 성공 확률 모델 평가

`main.ipynb`가 만든 시간순 OOF 예측을 평가합니다. 모델을 다시 학습하지 않으며, 주 평가는 해당 연도보다 앞선 OOF만으로 블렌딩·보정한 `pred_calibrated_walk_forward`입니다.

In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss, mean_squared_error, roc_auc_score

ROOT = Path.cwd().resolve()
RUN_DIR = Path(os.environ.get("BASEBALL_RUN_DIR", ROOT)).resolve()
ARTIFACT_DIR = RUN_DIR / "artifacts"
MODEL_DIR = RUN_DIR / "model"
OOF_PATH = ARTIFACT_DIR / "oof_predictions.parquet"
if not OOF_PATH.exists():
    raise FileNotFoundError("먼저 main.ipynb를 끝까지 실행해 주세요.")

oof = pd.read_parquet(OOF_PATH)
cv_metrics = pd.read_csv(ARTIFACT_DIR / "cv_metrics.csv")
importance = pd.read_csv(ARTIFACT_DIR / "feature_importance.csv")
feature_config = json.loads((MODEL_DIR / "feature_config.json").read_text(encoding="utf-8"))
print(f"OOF rows={len(oof):,}, seasons={sorted(oof['season'].unique())}")
display(cv_metrics.sort_values(["season", "model"]))


## 1. Fold별 확률 예측 지표

Brier가 주 지표입니다. Brier Skill은 해당 검증 시즌 이전 학습 라벨의 평균을 상수로 예측한 기준선과 비교합니다.

In [ ]:
TARGET = "control_success"
PREDICTION_COLUMNS = {
    "RMSE": "pred_rmse",
    "Logloss": "pred_logloss",
    "Walk-forward blend": "pred_blend_walk_forward",
    "Walk-forward calibrated": "pred_calibrated_walk_forward",
    "Final-fit calibrated (optimistic)": "pred_calibrated_final_fit",
}

def expected_calibration_error(y_true, prediction, bins=15):
    y_true = np.asarray(y_true)
    prediction = np.asarray(prediction)
    edges = np.linspace(0.0, 1.0, bins + 1)
    groups = np.clip(np.digitize(prediction, edges[1:-1]), 0, bins-1)
    ece = 0.0
    for index in range(bins):
        mask = groups == index
        if mask.any():
            ece += mask.mean() * abs(prediction[mask].mean() - y_true[mask].mean())
    return float(ece)

def season_prior(season):
    return float(feature_config["season_prior_map"][str(int(season))])

rows = []
for season, part in oof.groupby("season", sort=True):
    y = part[TARGET].to_numpy()
    prior = season_prior(season)
    baseline = mean_squared_error(y, np.full(len(y), prior))
    for label, column in PREDICTION_COLUMNS.items():
        prediction = np.clip(part[column].to_numpy(dtype="float64"), 1e-5, 1-1e-5)
        score = mean_squared_error(y, prediction)
        rows.append({
            "season": int(season), "model": label, "rows": len(part),
            "brier": score, "brier_skill_train_prior": 1-score/baseline,
            "log_loss": log_loss(y, prediction),
            "roc_auc": roc_auc_score(y, prediction),
            "ece_15": expected_calibration_error(y, prediction),
            "target_mean": y.mean(), "prediction_mean": prediction.mean(),
            "mean_gap": prediction.mean()-y.mean(),
        })
summary = pd.DataFrame(rows)
summary.to_csv(ARTIFACT_DIR / "evaluation_summary.csv", index=False, encoding="utf-8")
display(summary.style.format({"brier": "{:.6f}", "brier_skill_train_prior": "{:.4f}", "log_loss": "{:.6f}", "roc_auc": "{:.4f}", "ece_15": "{:.5f}", "target_mean": "{:.5f}", "prediction_mean": "{:.5f}", "mean_gap": "{:+.5f}"}))


## 2. 시즌 안정성과 calibration

In [ ]:
primary = summary[summary["model"] == "Walk-forward calibrated"].copy()
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
axes[0].plot(primary["season"], primary["brier"], marker="o")
axes[0].set(title="Walk-forward Brier", xlabel="season", ylabel="Brier (lower is better)")
axes[0].grid(alpha=0.3)
axes[1].plot(primary["season"], primary["target_mean"], marker="o", label="actual")
axes[1].plot(primary["season"], primary["prediction_mean"], marker="o", label="prediction")
axes[1].set(title="Mean probability drift", xlabel="season", ylabel="mean")
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[2].bar(primary["season"].astype(str), primary["ece_15"])
axes[2].set(title="ECE (15 bins)", xlabel="season", ylabel="ECE")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for axis, (season, part) in zip(axes, oof.groupby("season", sort=True)):
    prediction = part["pred_calibrated_walk_forward"].to_numpy()
    y = part[TARGET].to_numpy()
    bins = pd.qcut(prediction, q=10, duplicates="drop")
    reliability = pd.DataFrame({"prediction": prediction, "target": y, "bin": bins}).groupby("bin", observed=True).agg(prediction=("prediction", "mean"), actual=("target", "mean"), rows=("target", "size"))
    axis.plot([0.4, 0.65], [0.4, 0.65], linestyle="--", color="gray")
    axis.plot(reliability["prediction"], reliability["actual"], marker="o")
    axis.set(title=f"{season} reliability", xlabel="predicted", ylabel="actual")
    axis.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 3. 경기 유형·cold-start·이력 표본별 평가

In [ ]:
oof["pitcher_history_bin"] = pd.cut(
    oof["asof_pitcher_n"], bins=[-1, 0, 49, 199, 999, np.inf],
    labels=["0", "1-49", "50-199", "200-999", "1000+"],
)
oof["pitcher_cold_start"] = np.where(oof["asof_pitcher_n"] == 0, "cold", "seen")

def slice_scores(frame, columns, prediction_column="pred_calibrated_walk_forward"):
    records = []
    for keys, part in frame.groupby(columns, observed=True, sort=True):
        if len(part) < 100:
            continue
        keys = keys if isinstance(keys, tuple) else (keys,)
        prediction = np.clip(part[prediction_column], 1e-5, 1-1e-5)
        record = dict(zip(columns, keys))
        record.update(rows=len(part), brier=mean_squared_error(part[TARGET], prediction), target_mean=part[TARGET].mean(), prediction_mean=prediction.mean())
        records.append(record)
    return pd.DataFrame(records)

game_type_scores = slice_scores(oof, ["season", "game_type"])
history_scores = slice_scores(oof, ["season", "pitcher_history_bin"])
cold_start_scores = slice_scores(oof, ["season", "pitcher_cold_start"])
display(game_type_scores)
display(history_scores)
display(cold_start_scores)
game_type_scores.to_csv(ARTIFACT_DIR / "evaluation_by_game_type.csv", index=False, encoding="utf-8")
history_scores.to_csv(ARTIFACT_DIR / "evaluation_by_pitcher_history.csv", index=False, encoding="utf-8")


## 4. 피처 중요도와 최종 체크

In [ ]:
top = importance.sort_values("mean_importance", ascending=False).head(25).sort_values("mean_importance")
plt.figure(figsize=(9, 9))
plt.barh(top["feature"], top["mean_importance"])
plt.title("Top 25 mean CatBoost feature importance")
plt.tight_layout(); plt.show()

latest = primary.loc[primary["season"] == primary["season"].max()].iloc[0]
worst = primary.loc[primary["brier"].idxmax()]
print(f"latest season={int(latest['season'])}, Brier={latest['brier']:.6f}, mean gap={latest['mean_gap']:+.6f}")
print(f"worst season={int(worst['season'])}, Brier={worst['brier']:.6f}")
if abs(latest["mean_gap"]) > 0.01:
    print("주의: 최신 시즌 평균 calibration 오차가 1%p를 넘습니다.")
else:
    print("최신 시즌 평균 calibration 오차가 1%p 이내입니다.")
print("평가 표가 artifacts/evaluation_*.csv에 저장되었습니다.")
